### 1. Fashion-MNIST데이터셋 가져오기

In [ ]:
# Fashion MNIST 데이터셋 가져오기
from keras.datasets import fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

### 2. 탐색적 데이터 분석(EDA)

In [ ]:
train_images.shape, train_labels.shape, test_images.shape, test_labels.shape

In [ ]:
# 난수값을 생성해서 train_images와 train_labels중에 하나를 선택해서 이미지와 레이블을 출력한다.
import numpy as np
import matplotlib.pyplot as plt

index = np.random.randint(len(train_images))
image = train_images[index]
label = train_labels[index]

plt.imshow(image, cmap='gray')
plt.title(f"Label: {label}")
plt.axis('off')
plt.show()

In [ ]:
# train_images와 train_labels중에 클래스별로 2개씩 20개를 선택해서 이미지와 레이블을 출력한다.
# 이미지 출력시 2행 10열로 출력(같은 열에는 같은 클래스를 출력)
import numpy as np
import matplotlib.pyplot as plt

# Get unique labels
unique_labels = np.unique(train_labels);

# Prepare lists to store selected images and labels
selected_images = []
selected_labels = []

# Select 2 images per class
for label in unique_labels:
    # Find indices for the current label
    indices = np.where(train_labels == label)[0]
    # Select the first two images (or fewer if not enough are available)
    selected_images.extend(train_images[indices[:2]])
    selected_labels.extend(train_labels[indices[:2]])

# Create a 2x10 grid for plotting
plt.figure(figsize=(12, 3)) # Adjust figure size for better display

for col_idx in range(10): # For each of the 10 classes/columns
    # Plot the first image of the current class in the first row
    img1_index = col_idx * 2
    plt.subplot(2, 10, col_idx + 1) # row 1, col_idx + 1
    plt.imshow(selected_images[img1_index], cmap='gray')
    plt.title(f"Label: {selected_labels[img1_index]}")
    plt.axis('off')

    # Plot the second image of the current class in the second row
    img2_index = col_idx * 2 + 1
    plt.subplot(2, 10, 10 + col_idx + 1) # row 2, col_idx + 1 (10 is for the second row offset)
    plt.imshow(selected_images[img2_index], cmap='gray')
    plt.title(f"Label: {selected_labels[img2_index]}")
    plt.axis('off')

plt.tight_layout()
plt.show()

### 3. 데이터 전처리
3.1 정규화(Normalization) 0 ~ 255(uint8) -> 0 ~ 1(float) <br>
3.2 레이블을 10진수 -> one hot encoding <br>
3.3 이미지가 3차원(60000,28,28) -> 2차원(60000,28*28)

In [ ]:
train_images[1]

In [ ]:
# train_images와 test_images의 픽셀값을 0~1사이의 float로 Normalize
train_images_norm = train_images.astype('float32') / 255.0
test_images_norm = test_images.astype('float32') / 255.0

In [ ]:
train_images_norm[1]

In [ ]:
train_labels[0:10]

In [ ]:
# train_labels, test_labels를 one-hot encoding
from keras.utils import to_categorical

train_labels_onehot = to_categorical(train_labels)
test_labels_onehot = to_categorical(test_labels)

In [ ]:
train_labels_onehot[0:10]

In [ ]:
train_images_norm.shape, train_images_norm.shape[1], train_images_norm.shape[2]

In [ ]:
# train_images_norm과 test_images_norm 3차원에서 2차원으로 변환

# 이미지의 사이즈를 배열에서 읽어오기
image_size = train_images_norm.shape[1] * train_images_norm.shape[2]

train_images_2d = train_images_norm.reshape(-1, image_size)
test_images_2d = test_images_norm.reshape(-1, image_size)

print("Train images (flattened) shape:", train_images_2d.shape)
print("Test images (flattened) shape:", test_images_2d.shape)

### 4. 훈련(train)용 데이터셋에서 검증(valid)용 데이터셋 분할

In [ ]:
# y_train의 클래스별 갯수 확인
import numpy as np

# y_train은 one-hot encoded 상태이므로, 각 샘플의 실제 레이블을 찾기 위해 argmax를 사용합니다.
y_train_labels = np.argmax(train_labels_onehot, axis=1)

# 각 클래스별 갯수를 계산합니다.
class_counts = np.bincount(y_train_labels)

print("Class distribution in y_train:")
for i, count in enumerate(class_counts):
    print(f"Class {i}: {count} samples")

In [ ]:
from sklearn.model_selection import train_test_split

# train_images_2d를 8:2로 데이터셋 분할(train:valid)
X_train, X_val, y_train, y_val = train_test_split(
    train_images_2d, train_labels_onehot, test_size=0.2, random_state=42, stratify=train_labels_onehot,
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

In [ ]:
# y_train의 클래스별 갯수 확인
import numpy as np

# y_train은 one-hot encoded 상태이므로, 각 샘플의 실제 레이블을 찾기 위해 argmax를 사용합니다.
y_train_labels = np.argmax(y_train, axis=1)

# 각 클래스별 갯수를 계산합니다.
class_counts = np.bincount(y_train_labels)

print("Class distribution in y_train:")
for i, count in enumerate(class_counts):
    print(f"Class {i}: {count} samples")

In [ ]:
# y_train의 클래스별 갯수 확인
import numpy as np

# y_train은 one-hot encoded 상태이므로, 각 샘플의 실제 레이블을 찾기 위해 argmax를 사용합니다.
y_val_labels = np.argmax(y_val, axis=1)

# 각 클래스별 갯수를 계산합니다.
class_counts = np.bincount(y_val_labels)

print("Class distribution in y_val:")
for i, count in enumerate(class_counts):
    print(f"Class {i}: {count} samples")

### 5. 신경망 모델링

In [ ]:
num_classes = 10
hidden1_node = 512
hidden2_node = 128
hidden3_node = 64

In [ ]:
from keras import layers
from keras import Sequential

# Sequential 함수를 사용하여 히든 레이어의 갯수는 3개, 출력층의 노드의 갯수는 10개 구성
model = Sequential([
    layers.Input(shape=(image_size,)),  # Input layer with the flattened image size
    layers.Dense(hidden1_node, activation='relu'),  # First hidden layer
    layers.Dense(hidden2_node, activation='relu'),  # Second hidden layer
    layers.Dense(hidden3_node, activation='relu'),  # Third hidden layer
    layers.Dense(num_classes, activation='softmax')  # Output layer with 10 nodes for classification
])

model.summary()

### 6. 모델 설정(compile)

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

### 7. 콜백 함수 설정

In [ ]:
# 하이퍼-파라메터
EPOCHS = 100
BATCH_SIZE = 512
PATIENCE = 20
RLROP_PATIENCE = 15
RLROP_FACTOR = 0.5
RLROP_MIN_LR = 0.00001

### 콜백 함수 설명

*   **ModelCheckpoint**: `ModelCheckpoint` 콜백은 모델 학습 중 특정 조건(예: 검증 정확도가 가장 높을 때)에 따라 모델의 가중치를 저장하는 데 사용됩니다. `save_best_only=True`로 설정하면, 모니터링하는 지표(여기서는 `val_accuracy`)가 개선될 때만 모델이 저장되어 최적의 성능을 가진 모델을 유지할 수 있습니다.

*   **ReduceLROnPlateau**: `ReduceLROnPlateau` 콜백은 모델 학습 시 특정 지표(여기서는 `val_loss`)가 일정 기간 동안 개선되지 않을 때 학습률(Learning Rate)을 자동으로 감소시키는 기능을 합니다. `patience`는 개선이 없을 때까지 기다릴 에포크 수를, `factor`는 학습률을 줄이는 비율을 나타냅니다. `min_lr`은 학습률의 최솟값을 설정합니다. 이는 모델이 지역 최솟값에 갇히는 것을 방지하고 더 나은 수렴을 돕습니다.

*   **EarlyStopping**: `EarlyStopping` 콜백은 모델이 더 이상 개선되지 않을 때 학습을 조기에 중단하여 과적합을 방지하는 데 사용됩니다. `patience`는 모니터링하는 지표(여기서는 `val_accuracy`)가 개선되지 않을 때까지 기다릴 에포크 수를 의미합니다. 설정된 `patience` 기간 동안 지표가 개선되지 않으면 학습이 중단됩니다.

In [ ]:
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# ModelCheckpoint함수를 적용해서 val_accuracy를 기준으로 가장 좋은 성능의 모델을 mnist_best파일명으로 저장
MC_CB = ModelCheckpoint(filepath='F-mnist_best.keras', monitor='val_accuracy', save_best_only=True)

# ReduceLROnPlateu함수를 적용해서 val_loss를 기준으로 patience=15로 설정하고 factor=0.5로 설정
RLROP_CB = ReduceLROnPlateau(monitor='val_loss', factor=RLROP_FACTOR, patience=RLROP_PATIENCE, min_lr=RLROP_MIN_LR)

# EarlyStopping callback함수를 적용해서 patience=20, 모니터 기준은 val_accuracy로 설정
ES_CB = EarlyStopping(monitor='val_accuracy', patience=PATIENCE)

callbacks = [MC_CB, RLROP_CB, ES_CB]

### 8.모델 학습(fit)

In [ ]:
# callback함수를 사용하려면 callbacks인수에 리스트로 등록
# X_train, X_val, y_train, y_val
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks)

### 9. 학습 그래프(loss, accuracy) 출력

In [ ]:
# history를 사용해서 loss, accuracy, val_loss, val_accuracy의 그래프를 출력
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.tight_layout()
plt.show()

### 10. 모델 평가(evaluate)